In [0]:
# Check PySpark is working
spark.version

'4.2.0'

In [0]:
df = spark.read.table("flights")
print(df.count())

5819415


In [0]:
# See the columns and data types
df.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- DAY: long (nullable = true)
 |-- DAY_OF_WEEK: long (nullable = true)
 |-- FLIGHT_NUMBER: long (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: long (nullable = true)
 |-- DEPARTURE_TIME: long (nullable = true)
 |-- DEPARTURE_DELAY: long (nullable = true)
 |-- TAXI_OUT: long (nullable = true)
 |-- WHEELS_OFF: long (nullable = true)
 |-- SCHEDULED_TIME: long (nullable = true)
 |-- ELAPSED_TIME: long (nullable = true)
 |-- AIR_TIME: long (nullable = true)
 |-- DISTANCE: lo

In [0]:
from pyspark.sql.functions import col, avg, count, round, desc, when

# ── 1. Basic stats ──────────────────────────────────────────
print("=== Dataset Overview ===")
print(f"Total flights: {df.count():,}")
print(f"Cancelled flights: {df.filter(col('CANCELLED') == 1).count():,}")
print(f"Diverted flights: {df.filter(col('DIVERTED') == 1).count():,}")

# ── 2. Worst airlines by average arrival delay ──────────────
print("\n=== Top 10 Airlines by Average Arrival Delay (minutes) ===")
airline_delays = (
    df.filter(col('ARRIVAL_DELAY').isNotNull())
    .groupBy('AIRLINE')
    .agg(
        round(avg('ARRIVAL_DELAY'), 2).alias('AVG_ARRIVAL_DELAY'),
        count('*').alias('TOTAL_FLIGHTS')
    )
    .orderBy(desc('AVG_ARRIVAL_DELAY'))
)
display(airline_delays)

# ── 3. Worst routes by average departure delay ──────────────
print("\n=== Top 10 Worst Routes by Average Departure Delay ===")
route_delays = (
    df.filter(col('DEPARTURE_DELAY').isNotNull())
    .groupBy('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
    .agg(
        round(avg('DEPARTURE_DELAY'), 2).alias('AVG_DEPARTURE_DELAY'),
        count('*').alias('TOTAL_FLIGHTS')
    )
    .filter(col('TOTAL_FLIGHTS') > 100)
    .orderBy(desc('AVG_DEPARTURE_DELAY'))
    .limit(10)
)
display(route_delays)

# ── 4. Cancellations by reason ──────────────────────────────
print("\n=== Cancellation Reasons ===")
cancellations = (
    df.filter(col('CANCELLED') == 1)
    .groupBy('CANCELLATION_REASON')
    .agg(count('*').alias('COUNT'))
    .orderBy(desc('COUNT'))
)
display(cancellations)

# ── 5. Delay causes breakdown ───────────────────────────────
print("\n=== Average Delay by Cause (minutes) ===")
delay_causes = df.agg(
    round(avg('AIRLINE_DELAY'), 2).alias('Airline'),
    round(avg('WEATHER_DELAY'), 2).alias('Weather'),
    round(avg('AIR_SYSTEM_DELAY'), 2).alias('Air System'),
    round(avg('SECURITY_DELAY'), 2).alias('Security'),
    round(avg('LATE_AIRCRAFT_DELAY'), 2).alias('Late Aircraft')
)
display(delay_causes)

=== Dataset Overview ===
Total flights: 5,819,415
Cancelled flights: 89,884
Diverted flights: 15,187

=== Top 10 Airlines by Average Arrival Delay (minutes) ===


AIRLINE,AVG_ARRIVAL_DELAY,TOTAL_FLIGHTS
NK,14.47,115193
F9,12.5,90090
B6,6.68,262042
EV,6.59,554752
MQ,6.46,278791
OO,5.85,576814
UA,5.43,507762
VX,4.74,61248
WN,4.37,1242403
US,3.71,194223



=== Top 10 Worst Routes by Average Departure Delay ===


ORIGIN_AIRPORT,DESTINATION_AIRPORT,AVG_DEPARTURE_DELAY,TOTAL_FLIGHTS
JFK,EGE,46.98,107
ASE,DFW,40.46,280
ACY,DTW,37.39,145
EWR,PDX,33.84,322
SRQ,LGA,33.48,423
BQN,EWR,33.39,287
MVY,JFK,33.32,121
ACY,ORD,33.1,141
PIT,JFK,32.25,161
EGE,EWR,31.56,106



=== Cancellation Reasons ===


CANCELLATION_REASON,COUNT
B,48851
A,25262
C,15749
D,22



=== Average Delay by Cause (minutes) ===


Airline,Weather,Air System,Security,Late Aircraft
18.97,2.92,13.48,0.08,23.47


In [0]:
from pyspark.sql.functions import hour, lpad

# ── 6. Delays by day of week ────────────────────────────────
print("\n=== Average Arrival Delay by Day of Week ===")
day_map = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 
           4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}

day_delays = (
    df.filter(col('ARRIVAL_DELAY').isNotNull())
    .groupBy('DAY_OF_WEEK')
    .agg(round(avg('ARRIVAL_DELAY'), 2).alias('AVG_DELAY'))
    .orderBy('DAY_OF_WEEK')
)
display(day_delays)

# ── 7. Save results as Delta tables ────────────────────────
print("\nSaving results...")

airline_delays.write.mode("overwrite").saveAsTable("airline_delay_summary")
route_delays.write.mode("overwrite").saveAsTable("worst_routes_summary")
day_delays.write.mode("overwrite").saveAsTable("day_of_week_delay_summary")

print("Done! Results saved as Delta tables.")


=== Average Arrival Delay by Day of Week ===


DAY_OF_WEEK,AVG_DELAY
1,6.0
2,4.25
3,3.85
4,5.67
5,4.76
6,1.85
7,3.96



Saving results...
Done! Results saved as Delta tables.
